# Visualization 2 - Regional effect

In this Notebook, we will be be working with a data set of baby names in France. We will try to answer the following questions about the data :
- Is there a regional effect in the data? 
- Are some names more popular in some regions? 
- Are popular names generally popular across the whole country?

In [1]:
## Import libraries

import pandas as pd
import altair as alt
import json


# work-around to let Altair handle larger data sets
# This will use an external file to store our data instead of embedding it directly in the
# visualization
alt.data_transformers.enable('json')

pass # Don't show any output in this cell

Let's import and take a look at the data!

In [2]:
data = pd.read_csv('dpt2020.csv', sep=';')
#data = data.sample(frac=0.1)
data['dpt'] = data['dpt'].astype(str)

data.head()


,sexe,preusuel,annais,dpt,nombre
0,1,_PRENOMS_RARES,1900,02,7
1,1,_PRENOMS_RARES,1900,04,9
2,1,_PRENOMS_RARES,1900,05,8
3,1,_PRENOMS_RARES,1900,06,23
4,1,_PRENOMS_RARES,1900,07,9


Let's delete the "*_PRENOMS_RARES*" lines.

In [3]:
data = data[data["preusuel"] != "_PRENOMS_RARES"]


Let's import the geospatial data.

In [4]:
with open("departements.geojson", "r", encoding="utf-8") as f:
    geojson = json.load(f)


df_geo = pd.DataFrame([
    {
        "properties.code": feature["properties"]["code"],
        "properties.dep_name": feature["properties"]["nom"],
        "type": feature["type"],
        "geometry": feature["geometry"] # Dictionnaire GeoJSON pur préservé
    }
    for feature in geojson["features"]])

df_geo['properties.code'] = df_geo['properties.code'].astype(str)
df_geo[(df_geo['properties.code'] != '2A') & (df_geo['properties.code'] != '2B')]

df_geo.head(5) 

,properties.code,properties.dep_name,type,geometry
0,02,Aisne,Feature,"{'type': 'Polygon', 'coordinates': [[[3.172704..."
1,10,Aube,Feature,"{'type': 'Polygon', 'coordinates': [[[3.414788..."
2,14,Calvados,Feature,"{'type': 'Polygon', 'coordinates': [[[-1.11961..."
3,15,Cantal,Feature,"{'type': 'Polygon', 'coordinates': [[[2.508412..."
4,28,Eure-et-Loir,Feature,"{'type': 'Polygon', 'coordinates': [[[0.814824..."


Let's create a new Panda DataFrame, which represents the most given name in each department, for each year.

In [5]:
top_name = data.loc[data.groupby(["dpt", "annais"])["nombre"].idxmax()].copy()
top_name['dpt'] = top_name['dpt'].astype(str)
top_name.head(5)

,sexe,preusuel,annais,dpt,nombre
3049283,2,MARIE,1900,01,639
3049377,2,MARIE,1901,01,732
3049471,2,MARIE,1902,01,665
3049565,2,MARIE,1903,01,696
3049659,2,MARIE,1904,01,680


Sliders!

In [6]:
year_input = alt.binding_select(
    options=[x for x in range(1900, 2021)], 
    name='Year: '
)

year_param = alt.param(
    name="selected_year",
    value=2020,
    bind=year_input
)

''' 
year_slider = alt.binding_range(
    min=1900,
    max=2020,
    step=1,
    name="Select a year: "
)

year_param = alt.param(
    name="selected_year",
    value=1900,
    bind=year_slider
)
'''

' \nyear_slider = alt.binding_range(\n    min=1900,\n    max=2020,\n    step=1,\n    name="Select a year: "\n)\n\nyear_param = alt.param(\n    name="selected_year",\n    value=1900,\n    bind=year_slider\n)\n'

We create a base chart, fusing both the baby names dataset and the geospatial data.

In [ ]:
highlight = alt.selection_point(name="highlight", on="pointerover", empty=False)
select = alt.selection_point(name="select", on="click", fields=['dpt'])

chart = alt.Chart(data).add_params(
    year_param, highlight, select
).transform_filter(
    "datum.annais == selected_year"
).transform_window(
    rank='rank()',
    sort=[alt.SortField('nombre', order='descending')],
    groupby=['dpt']
).transform_filter(
    "datum.rank == 1"
).transform_lookup(
    lookup='dpt',
    from_=alt.LookupData(
        data=df_geo,
        key='properties\.code', 
        fields=['geometry', 'type', 'properties\.dep_name'],
    )
).mark_geoshape(
    strokeWidth=0.3,
    stroke="black",
).encode(
    color=alt.condition(
        select, 
        alt.Color("preusuel:N", title="Top Name", scale=alt.Scale(scheme='category20')),
        alt.value("white"), 
    ),
    opacity=alt.condition(highlight, alt.value(0.6), alt.value(1)),
    tooltip=[
        alt.Tooltip("properties\.dep_name:N", title="Department"),
        alt.Tooltip("preusuel:N", title="Most common name"),
        alt.Tooltip("nombre:Q", title="Nb of occurrences")
    ]
).properties(
    width=500, 
    height=400
).project(
    type='mercator'
)

chart

<>:18: SyntaxWarning: invalid escape sequence '\.'
<>:19: SyntaxWarning: invalid escape sequence '\.'
<>:32: SyntaxWarning: invalid escape sequence '\.'
<>:18: SyntaxWarning: invalid escape sequence '\.'
<>:19: SyntaxWarning: invalid escape sequence '\.'
<>:32: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_15626/3022374094.py:18: SyntaxWarning: invalid escape sequence '\.'
  key='properties\.code',
/tmp/ipykernel_15626/3022374094.py:19: SyntaxWarning: invalid escape sequence '\.'
  fields=['geometry', 'type', 'properties\.nom'],
/tmp/ipykernel_15626/3022374094.py:32: SyntaxWarning: invalid escape sequence '\.'
  alt.Tooltip("properties\.nom:N", title="Département"),
/usr/lib/python3/dist-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/usr/lib/py

alt.Chart(...)

In [19]:


diagramMostGivenNames = alt.Chart(data
    ).add_params(
        year_param,
        select
    ).transform_filter(
        "datum['annais'] == selected_year"
    ).transform_filter(
        "datum['dpt'] == select"
    ).transform_window(
        rank='rank()',
        sort=[alt.SortField('nombre', order='descending')]
    ).transform_filter(
        "datum.rank <= 30"
    ).mark_bar(
        size = 20
    ).encode(
        x=alt.X(
            'preusuel:N',
            sort="-y", 
        ),
        y=alt.Y('nombre:Q', title='Name')
    ).properties(width=alt.Step(30))

chart & diagramMostGivenNames

/usr/lib/python3/dist-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/usr/lib/python3/dist-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/usr/lib/python3/dist-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


OSError: [Errno 28] No space left on device

alt.VConcatChart(...)